In [69]:
import pandas as pd

In [70]:
df = pd.read_csv("DWBI_W1_Messy_Sales_Log.csv")

In [71]:
df.head()

,OrderID,Branch,Date,Product,Category,Qty,Amount
0,1001,LHR-Gulberg,12/09/2026,Cola 1.5L,Beverages,2,360
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
2,1003,KHI-Clifton,09-12-2026,Cola 1.5 Ltr,Beverages,3,540
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200


## Find at least six distinct problems 

## 1. Inconsistent Branch Naming (Fault 1)

In [72]:
# 1. Identify non-standard branch names using regex pattern
branch_fault_mask = df['Branch'].astype(str).str.contains(r'Lahore|LHR|khi|ISB', case=False, na=False)

# 2. Extract unique Order IDs and display matching records
fault_1_order_ids = df[branch_fault_mask]['OrderID'].unique()
print("Order IDs with Branch Naming Faults:", fault_1_order_ids)

# Display full rows for inspection
df[df['OrderID'].isin(fault_1_order_ids)].head(6)

Order IDs with Branch Naming Faults: [1001 1002 1003 1004 1005 1006 1007 1008 1009 1010 1011 1012 1013 1014
 1015 1016 1017 1018 1019 1020 1021 1022 1023 1024 1025 1026 1027 1028
 1029 1030 1031 1032 1033 1034 1035 1036 1037]


,OrderID,Branch,Date,Product,Category,Qty,Amount
0,1001,LHR-Gulberg,12/09/2026,Cola 1.5L,Beverages,2,360
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
2,1003,KHI-Clifton,09-12-2026,Cola 1.5 Ltr,Beverages,3,540
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
5,1005,khi clifton,12/09/2026,Fresh Milk 1L,NaN,2,440


## Why It Breaks the Report
When you run build a chart in Power BI, the database treats "LHR-Gulberg" and "Lahore Gulberg" as two completely different stores. This splits total revenue for the same physical branch across multiple rows, showing incorrect store-level performance and making branch comparisons unreliable.

## Assigned Owner
ETL / staging

## Justification
This can be consistently cleaned and standardized during data ingestion using deterministic lookup mapping tables or regex rules in staging

## 2.Duplicate Order Rows

In [73]:

# 1. Find rows where OrderID is duplicated
duplicate_mask = df.duplicated(subset=['OrderID'], keep=False)

# 2. Extract unique Order IDs and display matching records
fault_3_order_ids = df[duplicate_mask]['OrderID'].unique()
print("Order IDs with Duplicate Rows:", fault_3_order_ids)

# Display full duplicate rows
df[df['OrderID'].isin(fault_3_order_ids)]

Order IDs with Duplicate Rows: [1004]


,OrderID,Branch,Date,Product,Category,Qty,Amount
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200


## Why It Breaks the Report
Aggregations like SUM(Amount) or COUNT(OrderID) double-count duplicated rows. If order 1004 appears twice, the report adds its sales figure twice. This inflates overall revenue, profit metrics, and total order volume beyond actual business sales.

## Assigned Owner
Source system

## Justification
Primary key constraints, unique indexing, or UI submission blocking should be enforced at input time so duplicate transactions are never recorded in the first place.   

## 3. Inconsistent Product Naming

In [74]:

# 1. Identify rows where the product name uses codes or non-standard variations
product_fault_mask = df['Product'].astype(str).str.contains(r'CL-150|Ltr', case=False, na=False)

# 2. Extract unique Order IDs and display matching records
fault_4_order_ids = df[product_fault_mask]['OrderID'].unique()
print("Order IDs with Product Naming Faults:", fault_4_order_ids)

# Display full rows for inspection
df[df['OrderID'].isin(fault_4_order_ids)]

Order IDs with Product Naming Faults: [1002 1003 1021]


,OrderID,Branch,Date,Product,Category,Qty,Amount
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
2,1003,KHI-Clifton,09-12-2026,Cola 1.5 Ltr,Beverages,3,540
21,1021,KHI-Clifton,12 Sep 26,CL-150,Beverages,2,360


## Why It Breaks the Report
Product-level inventory and sales reports group metrics by Product.
Labeling the same item as "Cola 1.5L", "CL-150", and "Cola 1.5 Ltr" splits sales for a single product across three distinct inventory items. As a result, top-selling product rankings and stock reorder triggers become incorrect.

## Assigned Owner
The business

## Justification
Merging product codes like CL-150 and Cola 1.5L requires explicit human approval from product or inventory managers to define master data management rules

## 4. Category Inconsistent or Blank

In [75]:
# 1. Identify missing, empty, or non-standard category entries
category_fault_mask = (
    df['Category'].isna() | 
    (df['Category'].astype(str).str.strip().isin(['', 'Bev.']))
)

# 2. Extract unique Order IDs and display matching records
fault_5_order_ids = df[category_fault_mask]['OrderID'].unique()
print("Order IDs with Category Faults:", fault_5_order_ids)

# Display full rows for inspection
df[df['OrderID'].isin(fault_5_order_ids)]

Order IDs with Category Faults: [1002 1005 1015 1018 1025 1033]


,OrderID,Branch,Date,Product,Category,Qty,Amount
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
5,1005,khi clifton,12/09/2026,Fresh Milk 1L,NaN,2,440
15,1015,ISB-F7,09-12-2026,Orange Juice 1L,Bev.,3,960
18,1018,KHI-Clifton,2026-09-12,OJ-1L,Bev.,4,"1,280"
25,1025,ISB F-7,12 Sep 26,Orange Juice 1L,Bev.,2,640
33,1033,ISB F-7,09-12-2026,Milk 1 Litre,NaN,4,880


## Why It Breaks the Report
Missing or abbreviated category names create an unwanted (Blank) or Null bucket in BI dashboards.
Instead of showing total sales for "Beverages", revenue gets fragmented into separate buckets ("Beverages", "Bev.", and "(Blank)"), distorting high-level department executive summaries.

## Assigned Owner
The business

## Justification
Categorization logic and missing attribute assignments require business stakeholders to decide proper default classifications or product catalog definitions.   

## 5. Amounts Stored as Text

In [76]:
# 1. Identify rows where Amount contains non-numeric characters (currency text or commas)
amount_text_mask = df['Amount'].astype(str).str.contains(r'[^\d.]', na=False)

# 2. Extract unique Order IDs and display matching records
fault_6_order_ids = df[amount_text_mask]['OrderID'].unique()
print("Order IDs with Text Amount Faults:", fault_6_order_ids)

# Display full rows for inspection
df[df['OrderID'].isin(fault_6_order_ids)].head(5)

Order IDs with Text Amount Faults: [1002 1006 1007 1013 1017 1018 1023 1024 1032]


,OrderID,Branch,Date,Product,Category,Qty,Amount
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
6,1006,LHR-Gulberg,2026-09-12,Basmati Rice 5kg,Grocery,1,"2,450"
7,1007,ISB-F7,12/09/2026,LED Bulb 12W,Electrical,-1,-350
13,1013,Lahore Gulberg,12 Sep 26,Cooking Oil 5L,grocery,5,"9,450"
17,1017,Lahore Gulberg,09-12-2026,Salted Biscuits Pack,Snacks,1,Rs 120


## Why It Breaks the Report
Database engines cannot perform mathematical operations on text values like "Rs 180" or "2,450".
Executing SUM(Amount) on text columns results in an explicit query error, or the aggregation quietly ignores these values and converts them to 0 or Null, undercounting total revenue.

## Assigned Owner
ETL / staging

## Justification 
Text stripping rules can automatically purge currency symbols, clean commas, and cast the column into a numeric format during transformation

## 6. Negative Quantities and Amounts for Returns

In [77]:
# 1. Identify rows where Quantity or Amount are negative numbers
return_fault_mask = (
    (pd.to_numeric(df['Qty'], errors='coerce') < 0) | 
    (df['Amount'].astype(str).str.contains('-', na=False))
)

# 2. Extract unique Order IDs and display matching records
fault_7_order_ids = df[return_fault_mask]['OrderID'].unique()
print("Order IDs with Return/Negative Value Faults:", fault_7_order_ids)

# Display full rows for inspection
df[df['OrderID'].isin(fault_7_order_ids)]

Order IDs with Return/Negative Value Faults: [1007]


,OrderID,Branch,Date,Product,Category,Qty,Amount
7,1007,ISB-F7,12/09/2026,LED Bulb 12W,Electrical,-1,-350


## Why It Breaks the Report
If both quantity and amount are recorded as negative values (e.g., Qty = -1 and Amount = -180), calculating line total using Qty * Amount results in a positive value:$$\text{Line Total} = (-1) \times (-180) = +180$$This adds money to total sales instead of subtracting the returned item, artificially increasing calculated revenue. Standard data warehouse practice requires returns to be tracked as positive values with a dedicated transaction status flag or isolated into a separate returns table.

## Assigned Owner
Source system

## Justification
Point-of-Sale software should natively prevent double-negative entry by enforcing structured transaction types for returns at the recording layer

## Total Beverages Revenue

In [78]:


# 1. Deduplicate complete order rows
df_clean = df.drop_duplicates(subset=['OrderID']).copy()

# 2. Clean 'Amount' column (strip 'Rs', commas, and spaces)
df_clean['Amount'] = (
    df_clean['Amount']
    .astype(str)
    .str.replace(r'[^\d.-]', '', regex=True)
    .astype(float)
)

# 3. Fix Decimal/Pricing Anomaly (Order 1008 price fixed from 3600 to 360)
df_clean.loc[df_clean['OrderID'] == 1008, 'Amount'] = 360.0

# 4. Standardize Product Names (Map 'CL-150', 'Cola 1.5 Ltr' -> 'Cola 1.5L')
product_map = {
    'CL-150': 'Cola 1.5L',
    'Cola 1.5 Ltr': 'Cola 1.5L'
}
df_clean['Product'] = df_clean['Product'].replace(product_map)

# 5. Standardize & Impute Categories
# Step A: Map 'Bev.' to 'Beverages'
df_clean['Category'] = df_clean['Category'].replace({'Bev.': 'Beverages'})

# Step B: Impute missing/blank categories based on cleaned product name
beverage_products = ['Cola 1.5L']
df_clean.loc[
    (df_clean['Category'].isna()) | (df_clean['Category'].str.strip() == ''),
    'Category'
] = df_clean['Product'].apply(lambda x: 'Beverages' if x in beverage_products else np.nan)

# 6. Adjust Returns (Ensure return order 1007 subtracts revenue)
df_clean.loc[df_clean['OrderID'] == 1007, 'Amount'] = -abs(
    df_clean.loc[df_clean['OrderID'] == 1007, 'Amount'].values[0]
)

# 7. Calculate Total Beverages Revenue
beverages_df = df_clean[df_clean['Category'] == 'Beverages']
total_beverages_revenue = beverages_df['Amount'].sum()
print(" --------------------------------------------------------------")
print(f"Total Beverages Revenue: Rs. {total_beverages_revenue:,.2f}")
print(" --------------------------------------------------------------")
## beverages_df

 --------------------------------------------------------------
Total Beverages Revenue: Rs. 6,320.00
 --------------------------------------------------------------


## Confidence Level: Medium
To raise this confidence level to High, I would need access to the explicit master business rules (such as the official Data Governance catalog or Master Data Management guidelines) to confirm whether 1008 was an actual user entry typo or a valid bulk order pricing exception, and whether blank categories should be strictly imputed based on product maps or flagged for manual compliance review.

## Dashboard Loading Blockers
Loading this file directly into a dashboard fails because uncleaned text values in numeric metrics crash standard SQL aggregations, while inconsistent dimension naming fragments single entities across multiple visual categories.

<h2> Preventable Data Entry Problems </h2>
<h3> 1. Duplicate Order Prevention </h3>
<h4> Single Change: </h4>
Implement a Primary Key / Unique Constraint validation on OrderID directly in the database interface or Point-of-Sale (POS) application layer.
<h4> Why it works: </h4>
When a user or system attempts to submit an order with an OrderID that already exists, the application immediately blocks the submission and displays an entry error, preventing duplicate records from being written into the source database.   
<h3> 2. Inconsistent Branch Naming Prevention</h3>
<h4> Single Change: </h4> 
Replace the free-text input field for "Branch" with a restricted drop-down menu (UI Select Picker) driven by an enterprise master branch table.
<h4> Why it works: </h4>
Restricting users to selecting pre-approved, standardized branch options (e.g., "Lahore Gulberg") eliminates manual typing, abbreviations, casing mismatches, and spelling errors at the point of capture.